In [1]:
import pandas as pd
import numpy as np
import re
import os

In [2]:
EU_COUNTRIES = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

### AI adoption rate - 2023

The dependent variable is the proportion of firms within each country - sector using at least one type of AI in 2023. Unit of measurement - share of the firms. PC_ENT

In [16]:
ai_adopt = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/ain2.csv', index_col=0)
print(ai_adopt)

       freq size_emp nace_r2           indic_is    unit geo  num_2021  \
0         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  AT       NaN   
1         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BA       NaN   
2         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BE       NaN   
3         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  BG       NaN   
4         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT  CY       NaN   
...     ...      ...     ...                ...     ...  ..       ...   
221987    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  RO       NaN   
221988    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  RS       NaN   
221989    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SE       NaN   
221990    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SI       NaN   
221991    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT  SK       NaN   

       flag_2021  num_2023 flag_2023  num_2024 flag_2024  
0            NaN      8.24       NaN       NaN       NaN  
1    

In [17]:
un = pd.unique(ai_adopt['size_emp'])
print(un)

['GE10']


In [18]:
# Filter the data 
# E_AI_TANY - Enterprises use at least one of the AI technologies

ai_adopt_2023 = ai_adopt.copy()
ai_adopt_2023 = ai_adopt.drop(columns=[ 'freq', 'size_emp', 'num_2021', 'flag_2021', 'flag_2023', 'num_2024', 'flag_2024'])

ai_adopt_2023 = ai_adopt_2023[
    (ai_adopt_2023["indic_is"] == "E_AI_TANY") &
    (ai_adopt_2023["unit"] == "PC_ENT") 
]

ai_adopt_2023 = ai_adopt_2023[ai_adopt_2023['geo'].isin(EU_EFTA)]
print(ai_adopt_2023)

       nace_r2   indic_is    unit geo  num_2023
3809         C  E_AI_TANY  PC_ENT  AT     12.31
3811         C  E_AI_TANY  PC_ENT  BE     15.31
3812         C  E_AI_TANY  PC_ENT  BG      2.55
3813         C  E_AI_TANY  PC_ENT  CY      3.81
3814         C  E_AI_TANY  PC_ENT  CZ      6.01
...        ...        ...     ...  ..       ...
221156    S951  E_AI_TANY  PC_ENT  PT      8.82
221157    S951  E_AI_TANY  PC_ENT  RO      0.00
221159    S951  E_AI_TANY  PC_ENT  SE      6.67
221160    S951  E_AI_TANY  PC_ENT  SI       NaN
221161    S951  E_AI_TANY  PC_ENT  SK      0.00

[1339 rows x 5 columns]


In [19]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s)
    m = re.match(r'^([A-U])', s)
    if not m:
        return np.nan
    first = m.group(1)

    if re.search(r'-([A-U])', s) and re.search(r'-([A-U])', s).group(1) != first:
        return np.nan

    if re.search(r'_([A-U])', s) and re.search(r'_([A-U])', s).group(1) != first:
        return np.nan

    return first

ai_adopt_2023['nace_r2_1d'] = ai_adopt_2023['nace_r2'].map(nace_section_or_nan)
print(ai_adopt_2023)

       nace_r2   indic_is    unit geo  num_2023 nace_r2_1d
3809         C  E_AI_TANY  PC_ENT  AT     12.31          C
3811         C  E_AI_TANY  PC_ENT  BE     15.31          C
3812         C  E_AI_TANY  PC_ENT  BG      2.55          C
3813         C  E_AI_TANY  PC_ENT  CY      3.81          C
3814         C  E_AI_TANY  PC_ENT  CZ      6.01          C
...        ...        ...     ...  ..       ...        ...
221156    S951  E_AI_TANY  PC_ENT  PT      8.82          S
221157    S951  E_AI_TANY  PC_ENT  RO      0.00          S
221159    S951  E_AI_TANY  PC_ENT  SE      6.67          S
221160    S951  E_AI_TANY  PC_ENT  SI       NaN          S
221161    S951  E_AI_TANY  PC_ENT  SK      0.00          S

[1339 rows x 6 columns]


In [ ]:
ai_adopt_2023.drop(columns=['nace_r2', 'indic_is', 'unit'], inplace=True) 
ai_adopt_2023.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(ai_adopt_2023)

In [ ]:
ai_adopt_2023 = ai_adopt_2023.dropna()
print(ai_adopt_2023)

       geo  num_2023 nace_r2
3809    AT     12.31       C
3811    BE     15.31       C
3812    BG      2.55       C
3813    CY      3.81       C
3814    CZ      6.01       C
...     ..       ...     ...
221155  PL      9.33       S
221156  PT      8.82       S
221157  RO      0.00       S
221159  SE      6.67       S
221161  SK      0.00       S

[994 rows x 3 columns]


In [ ]:
ai_adopt_2023_agg = ai_adopt_2023.groupby(['geo', 'nace_r2'])['num_2023'].mean().reset_index()
ai_adopt_2023_agg.rename(columns={'num_2023': 'mn_ai_adopt'}, inplace=True)
print(ai_adopt_2023_agg)

In [30]:
print(ai_adopt_2023_agg)

    geo nace_r2  mn_ai_adopt
0    AT       C    14.751875
1    AT       D    29.510000
2    AT       E     7.160000
3    AT       F     4.280000
4    AT       G     8.320000
..   ..     ...          ...
297  SK       J    17.745000
298  SK       L     6.470000
299  SK       M    14.595000
300  SK       N    13.840000
301  SK       S     0.000000

[302 rows x 3 columns]


In [31]:
c = len(pd.unique(ai_adopt_2023_agg['geo']))
print(f'Number of the available countries for the ai adoption variable {c}')

n = len(pd.unique(ai_adopt_2023_agg['nace_r2']))
print(f'Number of the available nace codes for the ai adoption variable {n}')

print(f'Maximum number of the available country-sector rows for the ai adoption variable {c*n}')
print(f'Real number of the available country-sector rows for the ai adoption variable {len(ai_adopt_2023_agg.index)}')


Number of the available countries for the ai adoption variable 28
Number of the available nace codes for the ai adoption variable 12
Maximum number of the available country-sector rows for the ai adoption variable 336
Real number of the available country-sector rows for the ai adoption variable 302


### Wages

In [44]:
wages = pd.read_csv('C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/processed data/lc.csv')
print(wages)

        freq currency     unit sizeclas nace_r2 lcstruct geo      num_2016  \
0          A      EUR  P_SAL_H    10-49       B      D01  AL  2.500000e+00   
1          A      EUR  P_SAL_H    10-49       B      D01  AT  3.099000e+01   
2          A      EUR  P_SAL_H    10-49       B      D01  BA  3.910000e+00   
3          A      EUR  P_SAL_H    10-49       B      D01  BE  3.548000e+01   
4          A      EUR  P_SAL_H    10-49       B      D01  BG           NaN   
...      ...      ...      ...      ...     ...      ...  ..           ...   
1254268    A      PPS    TOTAL    TOTAL     S96    D1111  RS  3.134016e+07   
1254269    A      PPS    TOTAL    TOTAL     S96    D1111  SI  4.127085e+07   
1254270    A      PPS    TOTAL    TOTAL     S96    D1111  SK  3.383858e+07   
1254271    A      PPS    TOTAL    TOTAL     S96    D1111  TR  4.904631e+08   
1254272    A      PPS    TOTAL    TOTAL     S96    D1111  UK  4.946674e+09   

        flag_2016      num_2020 flag_2020  
0             NaN  

In [ ]:
# Filter the data 

wg_2016 = wages.copy()
wg_2016 = wg_2016[
    (wg_2016['currency'] == 'EUR') &
    (wg_2016['sizeclas'] == 'GE10') &  # 10 employees or more
    (wg_2016['unit'] == 'P_SAL_H') & # per employeein full-time equivalrnts, per hour
    (wg_2016['lcstruct'] == 'D111') # Wages and Salaries (excluding apprentices)
]

wg_2016 = wg_2016.drop(columns=['freq', 'currency', 'unit', 'sizeclas', 'lcstruct', 'num_2020', 'flag_2020', 'flag_2016'])
wg_2016 = wg_2016[wg_2016['geo'].isin(EU_EFTA)]

print(wg_2016)

      nace_r2 geo  num_2016
57688     A01  SI       NaN
57718     A02  SI       NaN
57748     A03  SI       NaN
57806       B  AT     28.31
57808       B  BE     29.48
...       ...  ..       ...
73080     S96  PT      6.69
73081     S96  RO      1.97
73083     S96  SE     18.54
73084     S96  SI     10.67
73085     S96  SK      4.81

[3254 rows x 3 columns]


In [46]:
wg_2020 = wages.copy()

wg_2020 = wages.copy()
wg_2020 = wg_2020[
    (wg_2020['currency'] == 'EUR') &
    (wg_2020['sizeclas'] == 'GE10') &  # 10 employees or more
    (wg_2020['unit'] == 'P_SAL_H') & # per employeein full-time equivalrnts, per hour
    (wg_2020['lcstruct'] == 'D111') # Wages and Salaries (excluding apprentices)
]

wg_2020 = wg_2020.drop(columns=['freq', 'currency', 'unit', 'sizeclas', 'lcstruct', 'num_2016', 'flag_2020', 'flag_2016'])
wg_2020 = wg_2020[wg_2020['geo'].isin(EU_EFTA)]

print(wg_2020)

      nace_r2 geo  num_2020
57688     A01  SI       NaN
57718     A02  SI       NaN
57748     A03  SI       NaN
57806       B  AT     28.62
57808       B  BE     31.33
...       ...  ..       ...
73080     S96  PT      8.09
73081     S96  RO      3.68
73083     S96  SE     19.97
73084     S96  SI     14.62
73085     S96  SK      6.96

[3254 rows x 3 columns]


In [ ]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s)
    m = re.match(r'^([A-U])', s)
    if not m:
        return np.nan
    first = m.group(1)

    if re.search(r'-([A-U])', s) and re.search(r'-([A-U])', s).group(1) != first:
        return np.nan

    if re.search(r'_([A-U])', s) and re.search(r'_([A-U])', s).group(1) != first:
        return np.nan

    return first

wg_2016['nace_r2_1d'] = wg_2016['nace_r2'].map(nace_section_or_nan)

wg_2016.drop(columns=['nace_r2'], inplace=True) 
wg_2016.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(wg_2016)

      geo  num_2016 nace_r2_1d
57688  SI       NaN          A
57718  SI       NaN          A
57748  SI       NaN          A
57806  AT     28.31          B
57808  BE     29.48          B
...    ..       ...        ...
73080  PT      6.69          S
73081  RO      1.97          S
73083  SE     18.54          S
73084  SI     10.67          S
73085  SK      4.81          S

[3254 rows x 3 columns]


In [49]:
wg_2020['nace_r2_1d'] = wg_2020['nace_r2'].map(nace_section_or_nan)

wg_2020.drop(columns=['nace_r2'], inplace=True) 
wg_2020.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(wg_2020)

      geo  num_2020 nace_r2
57688  SI       NaN       A
57718  SI       NaN       A
57748  SI       NaN       A
57806  AT     28.62       B
57808  BE     31.33       B
...    ..       ...     ...
73080  PT      8.09       S
73081  RO      3.68       S
73083  SE     19.97       S
73084  SI     14.62       S
73085  SK      6.96       S

[3254 rows x 3 columns]


In [50]:
wg_2016 = wg_2016.dropna()
wg_2016_agg = wg_2016.groupby(['geo', 'nace_r2'])['num_2016'].mean().reset_index()
wg_2016_agg.rename(columns={'num_2016': 'wg_2016'}, inplace=True)
print(wg_2016_agg)

    geo nace_r2    wg_2016
0    AT       B  36.046667
1    AT       C  26.161364
2    AT       D  39.230000
3    AT       E  22.808000
4    AT       F  25.222500
..   ..     ...        ...
520  SK       O   7.560000
521  SK       P   7.220000
522  SK       Q   6.710000
523  SK       R   5.732000
524  SK       S   5.290000

[525 rows x 3 columns]


In [51]:
wg_2020 = wg_2020.dropna()
wg_2020_agg = wg_2020.groupby(['geo', 'nace_r2'])['num_2020'].mean().reset_index()
wg_2020_agg.rename(columns={'num_2020': 'wg_2020'}, inplace=True)
print(wg_2020_agg)

    geo nace_r2    wg_2020
0    AT       B  36.436667
1    AT       C  30.918333
2    AT       D  43.390000
3    AT       E  27.446000
4    AT       F  26.825000
..   ..     ...        ...
524  SK       O  10.590000
525  SK       P  10.750000
526  SK       Q   9.605000
527  SK       R   9.128000
528  SK       S   6.745000

[529 rows x 3 columns]


In [54]:
print('WAGES 2016')
c_1 = len(pd.unique(wg_2016_agg['geo']))
print(f'Number of the available countries for the wages in 2016 variable {c_1}')

n_1 = len(pd.unique(wg_2016_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2016 variable {n_1}')

print(f'Maximum number of the available country-sector rows for the wages in 2016 variable {c_1*n_1}')
print(f'Real number of the available country-sector rows for the wages in 2016 variable {len(wg_2016_agg.index)}')

print('WAGES 2020')
c_2 = len(pd.unique(wg_2020_agg['geo']))
print(f'Number of the available countries for the wages in 2020 variable {c_2}')

n_2 = len(pd.unique(wg_2020_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2020 variable {n_2}')

print(f'Maximum number of the available country-sector rows for the wages in 2020 variable {c_2*n_2}')
print(f'Real number of the available country-sector rows for the wages in 2020 variable {len(wg_2020_agg.index)}')

WAGES 2016
Number of the available countries for the wages in 2016 variable 30
Number of the available nace codes for the wages in 2016 variable 18
Maximum number of the available country-sector rows for the wages in 2016 variable 540
Real number of the available country-sector rows for the wages in 2016 variable 525
WAGES 2020
Number of the available countries for the wages in 2020 variable 30
Number of the available nace codes for the wages in 2020 variable 18
Maximum number of the available country-sector rows for the wages in 2020 variable 540
Real number of the available country-sector rows for the wages in 2020 variable 529


### Labour cost

In [55]:
# Filter the data 

lc_2016 = wages.copy()

lc_2016 = lc_2016[
    (lc_2016['currency'] == 'EUR') &
    (lc_2016['sizeclas'] == 'GE10') &  # 10 employees or more
    (lc_2016['unit'] == 'P_SAL_H') & # per employeein full-time equivalents, per hour
    (lc_2016['lcstruct'] == 'D01') # Total labour costs (excluding apprentices)
]

lc_2016 = lc_2016.drop(columns=['freq', 'currency', 'unit', 'sizeclas', 'lcstruct', 'num_2020', 'flag_2020', 'flag_2016'])
lc_2016 = lc_2016[lc_2016['geo'].isin(EU_EFTA)]

print(lc_2016)

      nace_r2 geo  num_2016
57678     A01  SI       NaN
57708     A02  SI       NaN
57738     A03  SI       NaN
57760       B  AT     39.61
57762       B  BE     40.70
...       ...  ..       ...
73034     S96  PT      8.60
73035     S96  RO      2.46
73037     S96  SE     26.03
73038     S96  SI     12.39
73039     S96  SK      6.44

[3254 rows x 3 columns]


In [63]:
lc_2020 = wages.copy()

lc_2020 = lc_2020[
    (lc_2020['currency'] == 'EUR') &
    (lc_2020['sizeclas'] == 'GE10') &  # 10 employees or more
    (lc_2020['unit'] == 'P_SAL_H') & # per employeein full-time equivalents, per hour
    (lc_2020['lcstruct'] == 'D01') # Total labour costs (excluding apprentices)
]

lc_2020 = lc_2020.drop(columns=['freq', 'currency', 'unit', 'sizeclas', 'lcstruct', 'num_2016', 'flag_2020', 'flag_2016'])
lc_2020 = lc_2020[lc_2020['geo'].isin(EU_EFTA)]

print(lc_2020)

      nace_r2 geo  num_2020
57678     A01  SI       NaN
57708     A02  SI       NaN
57738     A03  SI       NaN
57760       B  AT     41.15
57762       B  BE     42.47
...       ...  ..       ...
73034     S96  PT      9.38
73035     S96  RO      3.85
73037     S96  SE     27.19
73038     S96  SI     15.83
73039     S96  SK      8.90

[3254 rows x 3 columns]


In [57]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s)
    m = re.match(r'^([A-U])', s)
    if not m:
        return np.nan
    first = m.group(1)

    if re.search(r'-([A-U])', s) and re.search(r'-([A-U])', s).group(1) != first:
        return np.nan

    if re.search(r'_([A-U])', s) and re.search(r'_([A-U])', s).group(1) != first:
        return np.nan

    return first

lc_2016['nace_r2_1d'] = lc_2016['nace_r2'].map(nace_section_or_nan)

lc_2016.drop(columns=['nace_r2'], inplace=True) 
lc_2016.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(lc_2016)

      geo  num_2016 nace_r2
57678  SI       NaN       A
57708  SI       NaN       A
57738  SI       NaN       A
57760  AT     39.61       B
57762  BE     40.70       B
...    ..       ...     ...
73034  PT      8.60       S
73035  RO      2.46       S
73037  SE     26.03       S
73038  SI     12.39       S
73039  SK      6.44       S

[3254 rows x 3 columns]


In [64]:
lc_2020['nace_r2_1d'] = lc_2020['nace_r2'].map(nace_section_or_nan)

lc_2020.drop(columns=['nace_r2'], inplace=True) 
lc_2020.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
print(lc_2020)

      geo  num_2020 nace_r2
57678  SI       NaN       A
57708  SI       NaN       A
57738  SI       NaN       A
57760  AT     41.15       B
57762  BE     42.47       B
...    ..       ...     ...
73034  PT      9.38       S
73035  RO      3.85       S
73037  SE     27.19       S
73038  SI     15.83       S
73039  SK      8.90       S

[3254 rows x 3 columns]


In [59]:
lc_2016 = lc_2016.dropna()
lc_2016_agg = lc_2016.groupby(['geo', 'nace_r2'])['num_2016'].mean().reset_index()
lc_2016_agg.rename(columns={'num_2016': 'lc_2016'}, inplace=True)
print(lc_2016_agg)

    geo nace_r2    lc_2016
0    AT       B  51.180000
1    AT       C  35.549545
2    AT       D  54.170000
3    AT       E  30.554000
4    AT       F  36.272500
..   ..     ...        ...
520  SK       O  10.360000
521  SK       P   9.820000
522  SK       Q   9.092500
523  SK       R   7.694000
524  SK       S   7.112500

[525 rows x 3 columns]


In [65]:
lc_2020 = lc_2020.dropna()
lc_2020_agg = lc_2020.groupby(['geo', 'nace_r2'])['num_2020'].mean().reset_index()
lc_2020_agg.rename(columns={'num_2020': 'lc_2020'}, inplace=True)
print(lc_2020_agg)

    geo nace_r2    lc_2020
0    AT       B  51.893333
1    AT       C  41.991667
2    AT       D  59.640000
3    AT       E  37.750000
4    AT       F  39.760000
..   ..     ...        ...
524  SK       O  14.430000
525  SK       P  14.650000
526  SK       Q  13.000000
527  SK       R  11.970000
528  SK       S   8.887500

[529 rows x 3 columns]


In [67]:
print('LABOUR COST 2016')
c_1 = len(pd.unique(lc_2016_agg['geo']))
print(f'Number of the available countries for the wages in 2016 variable {c_1}')

n_1 = len(pd.unique(lc_2016_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2016 variable {n_1}')

print(f'Maximum number of the available country-sector rows for the wages in 2016 variable {c_1*n_1}')
print(f'Real number of the available country-sector rows for the wages in 2016 variable {len(lc_2016_agg.index)}')

print('LABOUR COST 2020')
c_2 = len(pd.unique(lc_2020_agg['geo']))
print(f'Number of the available countries for the wages in 2020 variable {c_2}')

n_2 = len(pd.unique(lc_2020_agg['nace_r2']))
print(f'Number of the available nace codes for the wages in 2020 variable {n_2}')

print(f'Maximum number of the available country-sector rows for the wages in 2020 variable {c_2*n_2}')
print(f'Real number of the available country-sector rows for the wages in 2020 variable {len(lc_2020_agg.index)}')

LABOUR COST 2016
Number of the available countries for the wages in 2016 variable 30
Number of the available nace codes for the wages in 2016 variable 18
Maximum number of the available country-sector rows for the wages in 2016 variable 540
Real number of the available country-sector rows for the wages in 2016 variable 525
LABOUR COST 2020
Number of the available countries for the wages in 2020 variable 30
Number of the available nace codes for the wages in 2020 variable 18
Maximum number of the available country-sector rows for the wages in 2020 variable 540
Real number of the available country-sector rows for the wages in 2020 variable 529
